# ParkSight AI: Parking Congestion Intelligence Platform
**Tagline**: Detect illegal parking hotspots, quantify congestion impact, forecast tomorrow's risk, and recommend targeted enforcement actions.

This notebook implements the complete AI pipeline.

In [ ]:
import ast
import json
import math
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple

import folium
from folium.plugins import HeatMap
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

try:
    import hdbscan
except ImportError:
    hdbscan = None

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)




In [ ]:
# Configuration
EARTH_RADIUS_M = 6_371_000.0
DEFAULT_EPS_METERS = 25.0
DEFAULT_MIN_SAMPLES = 10
DEFAULT_DEDUPE_ROUND = 5
DEFAULT_MIN_CLUSTER_SIZE = 5

CONGESTION_WEIGHTS: Dict[str, float] = {
    "PARKING IN A MAIN ROAD": 10.0,
    "DOUBLE PARKING": 9.0,
    "PARKING NEAR ROAD CROSSING": 8.0,
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC": 7.0,
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE": 6.0,
    "NO PARKING": 5.0,
    "WRONG PARKING": 4.0,
}
DEFAULT_FALLBACK_WEIGHT = 4.0
PEAK_HOURS = {8, 9, 10, 17, 18, 19, 20, 21}




In [ ]:
# Utils
def _to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", utc=True)

def _parse_list_cell(x: Any) -> List[str]:
    if pd.isna(x):
        return []
    txt = str(x).strip()
    if not txt:
        return []
    try:
        v = ast.literal_eval(txt)
        if isinstance(v, list):
            return [str(i).strip().upper() for i in v if str(i).strip()]
        return [str(v).strip().upper()]
    except Exception:
        return [txt.upper()]

def _norm_str(x: Any, default: str = "UNKNOWN") -> str:
    if pd.isna(x):
        return default
    s = str(x).strip()
    return s if s else default

def _norm_vehicle_type(x: Any) -> str:
    return re.sub(r"\s+", " ", _norm_str(x, "UNKNOWN").upper())

def _ensure_column(df: pd.DataFrame, col: str, default: Any) -> pd.DataFrame:
    if col not in df.columns:
        df[col] = default
    return df

def normalize_series(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn = s.min()
    mx = s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)




In [ ]:
# MODULE 1: Data Quality Engine
def load_and_clean(input_path: str, dedupe_round: int = DEFAULT_DEDUPE_ROUND) -> pd.DataFrame:
    print("Loading data...")
    df = pd.read_csv(input_path).copy()

    for col, default in {
        "created_datetime": pd.NaT,
        "closed_datetime": pd.NaT,
        "latitude": np.nan,
        "longitude": np.nan,
        "validation_status": "unknown",
        "violation_type": "UNKNOWN",
        "vehicle_type": "UNKNOWN",
        "vehicle_number": np.nan,
        "police_station": "UNKNOWN",
        "junction_name": "UNKNOWN",
    }.items():
        _ensure_column(df, col, default)

    for c in ["created_datetime", "closed_datetime"]:
        df[c] = _to_dt(df[c])

    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["latitude", "longitude"]).copy()

    df["validation_status_norm"] = df["validation_status"].astype(str).str.strip().str.lower()
    approved_like = {"approved", "true", "1", "yes", "validated"}
    rejected_like = {"rejected", "false", "0", "no", "invalid"}

    df["is_approved"] = df["validation_status_norm"].isin(approved_like).astype(int)
    df["is_rejected"] = df["validation_status_norm"].isin(rejected_like).astype(int)

    df["violation_list"] = df["violation_type"].apply(_parse_list_cell)

    for c in ["vehicle_type", "police_station", "junction_name"]:
        df[c] = df[c].apply(_norm_str)

    df["vehicle_type"] = pd.Series(df["vehicle_type"].values, index=df.index).apply(_norm_vehicle_type)
    vehicle_num = pd.Series(df["vehicle_number"].values, index=df.index)
    df["vehicle_number_norm"] = vehicle_num.astype(str).str.strip().str.upper()
    df.loc[df["vehicle_number_norm"].isin(["NAN", "NONE", ""]), "vehicle_number_norm"] = np.nan

    df["location_key"] = df["latitude"].round(dedupe_round).astype(str) + "_" + df["longitude"].round(dedupe_round).astype(str)
    df["rounded_time"] = df["created_datetime"].dt.floor("5min")

    signature = df["violation_type"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
    df["dedupe_key"] = (
        df["vehicle_number_norm"].fillna("UNKNOWN") + "|" + df["location_key"] + "|" + df["rounded_time"].astype(str) + "|" + signature.astype(str)
    )

    df = df.drop_duplicates(subset=["dedupe_key"]).copy()

    # Repeat Offender logic
    vehicle_key = df["vehicle_number_norm"].fillna("UNKNOWN")
    df["vehicle_case_count"] = df.groupby(vehicle_key)["dedupe_key"].transform("size")
    df["repeat_vehicle_flag"] = (df["vehicle_case_count"] > 1).astype(int)

    df["hour"] = df["created_datetime"].dt.hour
    df["dayofweek"] = df["created_datetime"].dt.dayofweek
    df["date"] = df["created_datetime"].dt.date
    df["peak_hour"] = df["hour"].isin(PEAK_HOURS).astype(int)

    df["has_junction"] = (
        df["junction_name"].notna() & (df["junction_name"].astype(str).str.upper() != "NO JUNCTION")
    ).astype(int)

    df = df.explode("violation_list").copy()
    df["violation_list"] = df["violation_list"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
    df.loc[df["violation_list"] == "", "violation_list"] = "UNKNOWN"

    # Actual Congestion Proxy calculation
    df["congestion_weight"] = df["violation_list"].map(CONGESTION_WEIGHTS).fillna(DEFAULT_FALLBACK_WEIGHT)
    df["peak_hour_boost"] = np.where(df["peak_hour"] == 1, 1.2, 1.0)
    df["junction_boost"] = np.where(df["has_junction"] == 1, 1.15, 1.0)
    df["congestion_proxy"] = df["congestion_weight"] * df["peak_hour_boost"] * df["junction_boost"]

    return df




In [ ]:
# MODULE 2: Spatial Intelligence
def _choose_clusterer(eps_meters: float, min_samples: int, min_cluster_size: int):
    if hdbscan is not None:
        return ("hdbscan", hdbscan.HDBSCAN(
            min_cluster_size=max(min_cluster_size, min_samples),
            min_samples=min_samples,
            metric="haversine",
            cluster_selection_method="eom"
        ))
    eps_radians = eps_meters / EARTH_RADIUS_M
    return ("dbscan", DBSCAN(eps=eps_radians, min_samples=min_samples, metric="haversine"))

def build_hotspots(df: pd.DataFrame, eps_meters: float = DEFAULT_EPS_METERS, min_samples: int = DEFAULT_MIN_SAMPLES,
                   min_cluster_size: int = DEFAULT_MIN_CLUSTER_SIZE) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print("Building hotspots...")
    if df.empty:
        return df, pd.DataFrame()

    coords = np.radians(np.column_stack([df["latitude"].to_numpy(), df["longitude"].to_numpy()]))
    cluster_type, clusterer = _choose_clusterer(eps_meters, min_samples, min_cluster_size)

    try:
        labels = clusterer.fit_predict(coords)
    except Exception:
        eps_radians = eps_meters / EARTH_RADIUS_M
        clusterer = DBSCAN(eps=eps_radians, min_samples=min_samples, metric="haversine")
        labels = clusterer.fit_predict(coords)
        cluster_type = "dbscan"

    df = df.copy()
    df["cluster_id"] = labels
    clustered = df[df["cluster_id"] != -1].copy()

    if clustered.empty:
        return df, pd.DataFrame()

    def summarize(g: pd.DataFrame) -> pd.Series:
        violations = len(g)
        peak_hour_ratio = g["peak_hour"].mean()
        junction_ratio = g["has_junction"].mean()
        
        active_days = g["date"].nunique()
        active_weeks = g["created_datetime"].dt.isocalendar().week.nunique()
        
        approval_ratio = g["is_approved"].mean() if "is_approved" in g.columns else 0.0
        repeat_offender_ratio = g["repeat_vehicle_flag"].mean() if "repeat_vehicle_flag" in g.columns else 0.0
        
        # Congestion proxy
        cluster_congestion_proxy = g["congestion_proxy"].sum()
        
        # Hotspot Growth Score
        max_dt = g["created_datetime"].max()
        recent_7 = len(g[g["created_datetime"] >= (max_dt - pd.Timedelta(days=7))])
        prev_7 = len(g[(g["created_datetime"] < (max_dt - pd.Timedelta(days=7))) & (g["created_datetime"] >= (max_dt - pd.Timedelta(days=14)))])
        growth_rate = (recent_7 - prev_7) / max(prev_7, 1)
        
        # Best enforcement window
        peak_hour = int(g["hour"].mode().iloc[0]) if len(g["hour"].mode()) else 8
        best_window = f"{peak_hour:02d}:00 - {(peak_hour+3):02d}:00"

        return pd.Series({
            "violations": violations,
            "peak_hour_ratio": peak_hour_ratio,
            "junction_ratio": junction_ratio,
            "active_days": active_days,
            "active_weeks": active_weeks,
            "approval_ratio": approval_ratio,
            "repeat_offender_ratio": repeat_offender_ratio,
            "cluster_congestion_proxy": cluster_congestion_proxy,
            "recent_7_day_violations": recent_7,
            "prev_7_day_violations": prev_7,
            "growth_rate": growth_rate,
            "best_window": best_window,
            "center_lat": g["latitude"].mean(),
            "center_lon": g["longitude"].mean(),
            "top_violation": g["violation_list"].mode().iloc[0] if len(g["violation_list"].mode()) else "UNKNOWN",
            "top_police_station": g["police_station"].mode().iloc[0] if len(g["police_station"].mode()) else "UNKNOWN",
            "top_junction": g["junction_name"].mode().iloc[0] if len(g["junction_name"].mode()) else "UNKNOWN",
        })

    hotspots = clustered.groupby("cluster_id", sort=False).apply(summarize).reset_index()
    hotspots = hotspots[hotspots["violations"] >= min_cluster_size].copy()
    
    if hotspots.empty:
        return df, pd.DataFrame()

    # Calculate relative persistence across all hotspots
    max_act = hotspots["active_days"].max()
    hotspots["persistence_score"] = hotspots["active_days"] / (max_act if max_act > 0 else 1)
    
    # MODULE 3: Congestion Impact Score
    # 35% Volume, 20% Peak Hour, 15% Junction, 15% Persistence, 10% Repeat Offenders, 5% Approval
    # We now also integrate our cluster_congestion_proxy logically into the score by weighting it more carefully
    proxy_norm = normalize_series(hotspots["cluster_congestion_proxy"])
    
    hotspots["impact_score"] = (
        0.35 * proxy_norm +  # using the robust proxy instead of bare volume
        0.20 * hotspots["peak_hour_ratio"] +
        0.15 * hotspots["junction_ratio"] +
        0.15 * hotspots["persistence_score"] +
        0.10 * hotspots["repeat_offender_ratio"] +
        0.05 * hotspots["approval_ratio"]
    ) * 100.0
    
    # Ensure 0-100 bounds
    hotspots["impact_score"] = hotspots["impact_score"].clip(0, 100)
    
    def assign_tier(score):
        if score >= 80: return "Critical"
        if score >= 60: return "High"
        if score >= 40: return "Medium"
        return "Low"
    
    hotspots["risk_tier"] = hotspots["impact_score"].apply(assign_tier)

    hotspots = hotspots.sort_values(["impact_score", "violations"], ascending=False).reset_index(drop=True)
    return df, hotspots




In [ ]:
# MODULE 4: Forecast Engine
def prepare_daily_training_data(df: pd.DataFrame, hotspots: pd.DataFrame) -> pd.DataFrame:
    print("Preparing forecast features...")
    if "cluster_id" not in df.columns or df.empty or hotspots.empty:
        return pd.DataFrame()

    clustered = df[df["cluster_id"] != -1].copy()
    if clustered.empty: return pd.DataFrame()

    daily = (
        clustered.groupby(["cluster_id", "date"])
        .agg(daily_violations=("latitude", "count"))
        .reset_index()
        .sort_values(["cluster_id", "date"])
    )

    daily["date"] = pd.to_datetime(daily["date"])
    daily["dow"] = daily["date"].dt.dayofweek
    daily["dayofmonth"] = daily["date"].dt.day

    frames = []
    for _, g in daily.groupby("cluster_id"):
        g = g.sort_values("date").copy()
        g["lag_1"] = g["daily_violations"].shift(1)
        g["lag_3"] = g["daily_violations"].shift(3)
        g["lag_7"] = g["daily_violations"].shift(7)
        g["lag_14"] = g["daily_violations"].shift(14)
        g["lag_30"] = g["daily_violations"].shift(30)
        
        g["roll_7_mean"] = g["daily_violations"].shift(1).rolling(7, min_periods=1).mean()
        g["roll_30_mean"] = g["daily_violations"].shift(1).rolling(30, min_periods=1).mean()
        g["roll_7_median"] = g["daily_violations"].shift(1).rolling(7, min_periods=1).median()
        
        g["target_next_day"] = g["daily_violations"].shift(-1)
        frames.append(g)

    train_df = pd.concat(frames, ignore_index=True)
    return train_df.dropna(subset=["target_next_day"]).copy()

class DualEnsemble:
    def __init__(self, xgb_model, lgb_model):
        self.xgb = xgb_model
        self.lgb = lgb_model

    def predict(self, X):
        xgb_p = self.xgb.predict(X) if self.xgb else np.zeros(len(X))
        lgb_p = self.lgb.predict(X) if self.lgb else np.zeros(len(X))
        
        if self.xgb and self.lgb:
            return 0.6 * xgb_p + 0.4 * lgb_p
        if self.xgb: return xgb_p
        if self.lgb: return lgb_p
        return np.zeros(len(X))
        
    def predict_with_confidence(self, X):
        xgb_p = self.xgb.predict(X) if self.xgb else np.zeros(len(X))
        lgb_p = self.lgb.predict(X) if self.lgb else np.zeros(len(X))
        
        preds = self.predict(X)
        
        # Confidence logic
        if self.xgb and self.lgb:
            diff = np.abs(xgb_p - lgb_p)
            mx = np.maximum(np.maximum(xgb_p, lgb_p), 1.0)
            conf = 1.0 - (diff / mx)
            conf = np.clip(conf, 0.0, 1.0)
        else:
            conf = np.ones(len(X)) * 0.8  # Default if missing one model
            
        return preds, conf

def train_and_forecast(train_df: pd.DataFrame):
    print("Training forecast models...")
    features = [
        "dow", "dayofmonth", "lag_1", "lag_3", "lag_7", "lag_14", "lag_30",
        "roll_7_mean", "roll_30_mean", "roll_7_median"
    ]

    data = train_df.dropna(subset=features + ["target_next_day"]).copy()
    if len(data) < 2:
        return None, pd.DataFrame(), {"model": "ensemble", "mae": 0.0, "rmse": 0.0, "r2": 0.0, "train_rows": 0, "val_rows": 0}

    X, y = data[features], data["target_next_day"]

    xgb_model, lgb_model = None, None
    if XGB_AVAILABLE:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
        xgb_model.fit(X, y)
    if LGBM_AVAILABLE:
        lgb_model = LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
        lgb_model.fit(X, y)

    ensemble = DualEnsemble(xgb_model, lgb_model)

    # Compute validation metrics
    split_idx = int(len(data) * 0.8)
    X_train, y_train = X.iloc[:split_idx], y.iloc[:split_idx]
    X_val, y_val = X.iloc[split_idx:], y.iloc[split_idx:]
    
    val_preds = ensemble.predict(X_val)
    mae = float(mean_absolute_error(y_val, val_preds))
    rmse = float(np.sqrt(mean_squared_error(y_val, val_preds)))
    r2 = float(r2_score(y_val, val_preds))
    
    metrics = {
        "model": "ensemble",
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "train_rows": len(X_train),
        "val_rows": len(X_val)
    }

    # Forecast for the 'latest' date per cluster
    latest = train_df.sort_values(["cluster_id", "date"]).groupby("cluster_id").tail(1).copy()
    latest[features] = latest[features].fillna(0)
    
    preds, _ = ensemble.predict_with_confidence(latest[features])
    xgb_pred = ensemble.xgb.predict(latest[features]) if ensemble.xgb else preds
    lgbm_pred = ensemble.lgb.predict(latest[features]) if ensemble.lgb else preds
    
    forecast_margin = np.maximum(1, np.abs(xgb_pred - lgbm_pred).round()).astype(int)
    predicted_violations = np.maximum(0, preds).round().astype(int)
    
    forecast_confidence = np.maximum(
        50.0,
        100.0 - (
            forecast_margin /
            np.maximum(predicted_violations, 1.0)
        ) * 100.0
    )
    
    latest["predicted_violations"] = predicted_violations
    latest["forecast_margin"] = forecast_margin
    latest["forecast_confidence"] = forecast_confidence.round(1)
    latest["latest_violations"] = latest["daily_violations"]
    
    return ensemble, latest[["cluster_id", "predicted_violations", "forecast_margin", "forecast_confidence", "latest_violations"]], metrics






In [ ]:
# MODULE 5: Enforcement Command Center & Formatting
def compute_resources(row):
    score = row.get("impact_score", 0)
    peak_ratio = row.get("peak_hour_ratio", 0)
    
    if score > 90:
        officers = 3
        tow = 2
    elif score > 75:
        officers = 2
        tow = 1
    else:
        officers = 1
        tow = 0
        
    actions = [f"{officers} Traffic Officer{'s' if officers > 1 else ''}"]
    if tow > 0:
        actions.append(f"{tow} Tow Vehicle{'s' if tow > 1 else ''}")
        
    if peak_ratio > 0.5:
        actions.append("Peak Hour Patrol")
        
    return ", ".join(actions)

def compute_forecast_risk(row):
    # Determine the risk level of tomorrow based on current score + expected increase
    score = row.get("impact_score", 0)
    inc = row.get("expected_increase_pct", 0)
    if score > 80 or (score > 60 and inc > 20):
        return "Critical"
    if score > 60 or (score > 40 and inc > 15):
        return "High"
    if score > 40 or inc > 10:
        return "Medium"
    return "Low"

def generate_explainability(row):
    rank = row.name + 1
    vol = int(row['violations'])
    peak = int(row['peak_hour_ratio'] * 100)
    growth = row.get('growth_rate', 0) * 100
    proxy = int(row.get('cluster_congestion_proxy', 0))
    
    return (f"Ranked #{rank} because:\n"
            f"• Traffic Obstruction Proxy: {proxy:,}\n"
            f"• {vol:,} total violations\n"
            f"• {peak}% peak-hour concentration\n"
            f"• Growth Rate: {'+' if growth > 0 else ''}{growth:.1f}%")

def compile_final_hotspots(hotspots, forecasts):
    print("Compiling final intelligence...")
    if hotspots.empty: return pd.DataFrame()
    
    df = hotspots.copy()
    if not forecasts.empty:
        df = df.merge(forecasts, on="cluster_id", how="left")
        df["predicted_violations"] = df["predicted_violations"].fillna(0)
        df["forecast_margin"] = df["forecast_margin"].fillna(1).astype(int)
        df["forecast_confidence"] = df["forecast_confidence"].fillna(50.0)
        df["latest_violations"] = df["latest_violations"].fillna(1)
        
        # Expected Increase
        df["expected_increase_pct"] = ((df["predicted_violations"] - df["latest_violations"]) / df["latest_violations"].clip(lower=1)) * 100
        df["forecast_risk"] = df.apply(compute_forecast_risk, axis=1)
    else:
        df["predicted_violations"] = 0
        df["forecast_margin"] = 1
        df["forecast_confidence"] = 50.0
        df["expected_increase_pct"] = 0
        df["forecast_risk"] = "Low"
        
    df["recommended_resources"] = df.apply(compute_resources, axis=1)
    
    # Sort again to ensure rank is correct
    df = df.sort_values(["impact_score", "predicted_violations"], ascending=False).reset_index(drop=True)
    df["why_ranked"] = df.apply(generate_explainability, axis=1)
    
    return df


def build_map(df: pd.DataFrame, hotspots: pd.DataFrame, out_path: str) -> None:
    print("Generating static hotspots HTML map...")
    if df.empty or "latitude" not in df.columns or "longitude" not in df.columns:
        return

    m = folium.Map(location=[df["latitude"].mean(), df["longitude"].mean()], zoom_start=12, tiles="cartodbpositron")
    HeatMap(df[["latitude", "longitude"]].dropna().values.tolist(), radius=12, blur=18, min_opacity=0.35).add_to(m)

    if hotspots is None or hotspots.empty:
        m.save(out_path)
        return

    for _, row in hotspots.head(50).iterrows():
        score = row.get("impact_score", 0.0)
        popup = folium.Popup(
            html=(
                f"<b>Cluster:</b> {int(row.get('cluster_id', -1))}<br>"
                f"<b>Violations:</b> {int(row.get('violations', 0))}<br>"
                f"<b>Impact Score:</b> {float(score):.2f}<br>"
                f"<b>Action:</b> {row.get('recommended_resources', 'Standard patrol')}<br>"
                f"<b>Top violation:</b> {row.get('top_violation', 'UNKNOWN')}<br>"
                f"<b>Top junction:</b> {row.get('top_junction', 'UNKNOWN')}"
            ),
            max_width=360,
        )
        tier = row.get("risk_tier", "Low")
        color = {"Critical": "red", "High": "orange", "Medium": "gold", "Low": "green"}.get(tier, "blue")
        folium.CircleMarker(
            location=[row.get("center_lat", df["latitude"].mean()), row.get("center_lon", df["longitude"].mean())],
            radius=float(min(20, max(6, float(row.get("violations", 0)) / 500))),
            popup=popup,
            tooltip=f"Impact: {float(score):.1f} | Tier: {tier}",
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
        ).add_to(m)

    m.save(out_path)


def main():
    INPUT_CSV = "parking_violations.csv"
    OUTDIR = Path("outputs")
    OUTDIR.mkdir(exist_ok=True)

    if not Path(INPUT_CSV).exists():
        print(f"File {INPUT_CSV} not found. Ensure the dataset is present.")
        return

    # 1. Clean
    df = load_and_clean(INPUT_CSV)
    
    # 2 & 3. Hotspots & Impact
    df, hotspots = build_hotspots(df)
    
    # 4. Forecast
    train_df = prepare_daily_training_data(df, hotspots)
    model, forecasts, metrics = train_and_forecast(train_df)
    
    # 5. Final Compilation
    final_hotspots = compile_final_hotspots(hotspots, forecasts)
    
    # Station ranking
    if "top_police_station" in final_hotspots.columns:
        station_ranking = (
            final_hotspots.groupby("top_police_station")
            .agg(
                total_hotspots=("cluster_id", "count"),
                avg_impact=("impact_score", "mean"),
                critical_count=("risk_tier", lambda x: (x=="Critical").sum())
            )
            .reset_index()
            .sort_values("avg_impact", ascending=False)
        )
        station_ranking.to_csv(OUTDIR / "station_ranking.csv", index=False)

    # Save
    final_hotspots.to_csv(OUTDIR / "hotspots.csv", index=False)
    forecasts.to_csv(OUTDIR / "predictions.csv", index=False)
    final_hotspots.to_csv(OUTDIR / "leaderboard_priority.csv", index=False)
    
    with open(OUTDIR / "model_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    build_map(df, final_hotspots, str(OUTDIR / "parking_hotspots_map.html"))
    
    # Create map data
    df.to_csv(OUTDIR / "clean_violations.csv", index=False)
    
    print("\n=== ParkSight AI Execution Complete ===")
    print(f"Total Hotspots Detected: {len(final_hotspots)}")
    print(f"Critical Hotspots: {(final_hotspots['risk_tier'] == 'Critical').sum()}")
    print(f"Outputs saved to {OUTDIR}/")

if __name__ == "__main__":
    main()
